# Lab Tasks - Solutions

In this lab, we will explore the analysis of a **dynamic network**, where relationships between nodes change over time. The dataset used for these tasks represents anonymised loan records between users on Prosper, a peer-to-peer lending platform. 

The raw data is stored in CSV format, where each row represents a record indicating who loaned money to whom. Each record includes a numeric timestamp, enabling temporal analysis of lending patterns and network evolution.

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

### Task 1

Load the loan record data from the file *loans.csv* into a Pandas DataFrame. Based on the timestamp data, divide the records into 10 separate time windows of equal duration. Each time window should be represented by a separate DataFrame containing the loan records for that time period.

In [ ]:
# load the data
df = pd.read_csv("loans.csv")
print(f"Read {len(df)} loan records")
df.head(10)

In [ ]:
# check the range of the data
min_time = df["time"].min()
max_time = df["time"].max()
print(f"Time range: {min_time} to {max_time}")

In [ ]:
# based on the timestamp data, divide the records into 10 separate time windows of equal duration:
num_windows = 10
window_size = round((max_time-min_time)/num_windows)
print(f"Window size: {window_size}")

In [ ]:
window_frames = []
current_min = min_time
for window_number in range(1, num_windows+1):
    # set the current window range
    win_min = current_min
    win_max = win_min + window_size
    # handle the last window correctly, so we don't lose data
    if window_number == num_windows:
        wf = df[(df["time"]>=win_min) & (df["time"]<=win_max)]
    else:
        # otherwise we exclude the last day
        wf = df[(df["time"]>=win_min) & (df["time"]<win_max)]
    window_frames.append(wf)
    print(f"Window {window_number:02d}: time {win_min} to {win_max-1} - {len(wf)} interactions")
    # increment
    current_min += window_size

### Task 2

For each time window, construct a corresponding unweighted directed time window network. In these networks, an edge represents a lending relationship, indicating who loaned money to whom during that time period.

In [ ]:
window_networks = []
for w in range(num_windows):
    # NB: this is a directed network
    g = nx.from_pandas_edgelist(window_frames[w], "from_id", "to_id", create_using=nx.DiGraph())  
    print(f"Window {w+1:02d} network has {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")
    window_networks.append(g)

### Task 3

From the time window networks constructed in Task 2, generate time series data for two network characterisation measures:

1. Network density
2. Average node degree

Examine these time series to see if there are any fundamental changes in the dynamic network structure over time. Do the lending patterns exhibit temporal trends or remain relatively stable across the observation period?

In [ ]:
# define a convenience function for plotting a time series using Pandas
def gen_ts_plot(values, measure_name, color):
    s_values = pd.Series(values)
    ax = s_values.plot(figsize=(10, 5), style=".-", ms=12, fontsize=12, color=color, linewidth=2, zorder=3)
    ax.set_xlabel("Time Window", fontsize=12)
    ax.set_ylabel(measure_name, fontsize=12)
    ax.set_title(f"{measure_name} over time", fontsize=12)
    ax.set_xlim((1, len(values)))
    ax.set_ylim((0, s_values.max()*1.1))
    ax.xaxis.grid()
    plt.show()

In [ ]:
# calculate the average degree for each time window network
values = {}
for w in range(num_windows):
    degrees = dict(window_networks[w].degree()).values()
    # get the mean
    values[w+1] = sum(degrees)/len(degrees)
# produce a plot of the values
gen_ts_plot(values, "Average degree", "blue")

In [ ]:
# calculate the density for each time window network
values = {}
for w in range(num_windows):
    values[w+1] = nx.density(window_networks[w])
# produce a plot of the values
gen_ts_plot(values, "Density", "darkred")